# **Start Section:**


In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/scour_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d07_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # scour train
    shutil.copy(glob.glob("/kaggle/input/**/d07_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # scour test
    print("Kaggle: scour data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: scour data copied to ./drive/MyDrive/Regression/scour_uncertainty_analysis/data


In [2]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel.
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 bayesian-optimization==3.2.0 properscoring==0.1 XlsxWriter==3.2.9 mapie==0.6.0 puncc==0.8.0
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
import ngboost
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from optuna.samplers import BaseSampler
from optuna.samplers import GridSampler
from optuna.samplers import TPESampler
from optuna.samplers import PartialFixedSampler
from optuna.samplers import CmaEsSampler
from optuna.samplers import QMCSampler
from optuna.samplers import NSGAIIISampler
from optuna.samplers import NSGAIISampler
from optuna.samplers import BruteForceSampler
from optuna.samplers import GPSampler
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from openpyxl.drawing.image import Image
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from sklearn.model_selection import KFold
from PIL import Image as PImage
from openpyxl.utils.dataframe import dataframe_to_rows
from pytorch_tabnet import TabNetRegressor
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


In [4]:
# Go to find & replace button and replace (scour_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.


In [5]:
feature_names = ['Ps', 'Pw', ' Skew  ', ' Velocity  ', ' Depth  ', ' D50  ', ' Gradation  ']
TARGET_LABEL = 'Scour depth'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)


In [6]:
train_data_path = "./drive/MyDrive/Regression/scour_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/scour_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")


Training data loaded successfully.
Test data loaded successfully.


In [7]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))



Shape of training data: (154, 8)
First 5 rows of training data:
     Ps   Pw   Skew     Velocity     Depth     D50     Gradation     Scour  
0  0.7  1.5        0          2.9       6.6   70.00           1.2       0.6
1  0.7  1.5        0          3.0       5.3   70.00           1.2       0.6
2  1.3  3.3        0          0.8       4.8    0.48           1.8       0.4
3  1.0  4.3        0          2.9       9.7    0.30           1.4       3.7
4  0.7  0.5       20          1.4       1.8    1.10           3.5       0.3

Shape of test data: (78, 8)
First 5 rows of test data:
     Ps   Pw   Skew     Velocity     Depth     D50     Gradation     Scour  
0  1.3  0.3       16          1.0       0.3    0.94           3.0       0.4
1  1.0  0.8        0          0.2       1.5    0.25           8.0       0.2
2  1.0  0.6       10          1.6       6.6    0.90           4.2       1.1
3  1.3  1.2        0          0.8       3.1    0.38           2.3       1.6
4  1.0  0.8        0          1.1       1

In [8]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)



Shape of X_train: (154, 7)
Shape of y_train: (154,)
Shape of X_test: (78, 7)
Shape of y_test: (78,)


In [9]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])



First five rows of normalized X_train:
[[-1.30505288 -0.05072804 -0.49867969  1.41932674  0.51129506  1.91265145
  -0.75772712]
 [-1.30505288 -0.05072804 -0.49867969  1.53187891  0.18675076  1.91265145
  -0.75772712]
 [ 1.56606345  1.51169548 -0.49867969 -0.94426887  0.06192603 -0.69389788
  -0.5721613 ]
 [ 0.13050529  2.37970854 -0.49867969  1.41932674  1.28520839 -0.70064672
  -0.69587185]
 [-1.30505288 -0.9187411   0.57841248 -0.26895584 -0.68702234 -0.6706519
  -0.04639146]]

First five rows of normalized X_test:
[[ 1.56606345 -1.09234371  0.36299405 -0.71916453 -1.06149653 -0.67665086
  -0.20102964]
 [ 0.13050529 -0.65833718 -0.49867969 -1.6195819  -0.76191718 -0.70252139
   1.34535224]
 [ 0.13050529 -0.83193979  0.0398664  -0.0438515   0.51129506 -0.6781506
   0.17010201]
 [ 1.56606345 -0.31113196 -0.49867969 -0.94426887 -0.36247805 -0.69764723
  -0.41752311]
 [ 0.13050529 -0.65833718 -0.49867969 -0.60661235 -0.71198729  1.53771627
  -0.38659547]]


# **Functions:**


In [10]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [11]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'TabNet' : 'TabNet',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']


In [12]:
# FIX M4: one miscoverage level for the conformal notebooks. AlphaNet LEARNS its own level alpha~,
# so ALPHA is only the nominal reference value reported next to it.
ALPHA = 0.1


# FIX A1: lambda is the size/coverage trade-off chosen by the user (Gauthier, Bach & Jordan 2026,
# eq. 9). The official code reports every lambda separately and never compares losses across
# lambdas. This table evaluates, for EVERY lambda, its best run (lowest final training loss within
# that lambda) on the test set, so the trade-off can be seen.
def _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test, y_test):
    rows = []
    for lam, res in sorted(all_results.items(), key=lambda kv: kv[0]):
        final_losses = res["all_losses"][:, -1]
        r = int(np.argmin(final_losses))
        ev = evaluate_acp(model=model, alpha_net=res["models"][r], X_calib=X_calib, y_calib=y_calib,
                          X_test=X_test, y_test=y_test)
        a = ev.get("alpha_hat", ev.get("alpha"))
        rows.append({
            "lambda": lam,
            "best_run (1-based)": r + 1,
            "final_train_loss": float(final_losses[r]),
            "alpha_learned": a,
            "Target_coverage": 1.0 - a,
            "test_coverage": ev["coverage"],
            "width": ev["interval_width"],
            "Miss_over_alpha": (1.0 - ev["coverage"]) / a,
        })
    return pd.DataFrame(rows)


In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def _to_numpy_2d(x):
    arr = x.to_numpy() if hasattr(x, "to_numpy") else np.asarray(x)
    if arr.ndim == 1:
        arr = arr.reshape(-1, 1)
    return arr.astype(np.float32)

def _to_numpy_1d(y):
    arr = y.to_numpy() if hasattr(y, "to_numpy") else np.asarray(y)
    return arr.reshape(-1).astype(np.float32)

def _ensure_excel_workbook(excel_file_path):
    if not excel_file_path:
        return
    os.makedirs(os.path.dirname(excel_file_path), exist_ok=True)
    if not os.path.exists(excel_file_path):
        wb = Workbook()
        wb.active.title = "Init"
        wb.save(_ensure_parent_dir(excel_file_path))

def save_plot_to_excel(fig, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    with io.BytesIO() as buf:
        fig.savefig(buf, format="png", bbox_inches="tight")
        buf.seek(0)
        img = Image(PImage.open(buf))
        workbook = load_workbook(_ensure_excel_file(excel_file_path))
        base_name = sheet_name
        i = 1
        while sheet_name in workbook.sheetnames:
            sheet_name = f"{base_name}_{i}"
            i += 1
        worksheet = workbook.create_sheet(title=sheet_name)
        worksheet.add_image(img, "A1")
        workbook.save(_ensure_parent_dir(excel_file_path))

def save_values_to_excel(results_dict, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    df = pd.DataFrame(results_dict)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

def save_dataframe_to_excel(df, excel_file_path, sheet_name):
    _ensure_excel_workbook(excel_file_path)
    workbook = load_workbook(_ensure_excel_file(excel_file_path))
    base_name = sheet_name
    i = 1
    while sheet_name in workbook.sheetnames:
        sheet_name = f"{base_name}_{i}"
        i += 1
    worksheet = workbook.create_sheet(title=sheet_name)
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=True), 1):
        for c_idx, value in enumerate(row, 1):
            worksheet.cell(row=r_idx, column=c_idx, value=value)
    workbook.save(_ensure_parent_dir(excel_file_path))

class AlphaNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )
        # FIX A2: start the output near alpha~ = 1 (sigmoid(5) ~ 0.993), as in the official code
        # (github.com/GauthierE/adaptive-coverage-policies, initialize_to_one), so alpha~ starts in the
        # region alpha~ > 1/(n+1) where the width formula is valid. Was: random initial output.
        with torch.no_grad():
            self.net[2].weight.data.normal_(0, 0.01)
            self.net[2].bias.data.fill_(5.0)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def interval_size(scores, alpha, eps=1e-6):
    n = scores.numel()
    denominator = torch.clamp(alpha * (n + 1) - 1.0, min=eps)
    return 2.0 * scores.sum() / denominator

def _compute_scores_without_index(base_model, X_calib, y_calib, holdout_idx):
    loo_X = np.delete(X_calib, holdout_idx, axis=0)
    loo_y = np.delete(y_calib, holdout_idx, axis=0)
    return np.abs(base_model.predict(loo_X) - loo_y)

def _build_loo_feature_matrix(base_model, X_calib, y_calib):
    rows = []
    for idx in range(len(X_calib)):
        scores = _compute_scores_without_index(base_model, X_calib, y_calib, idx)
        rows.append([scores.sum()])
    return torch.tensor(np.asarray(rows, dtype=np.float32))

def _resolve_model_params(model_class, best_params):
    params = dict(best_params or {})
    if "TabNet" in model_class.__name__ and "verbose" in params:
        params.pop("verbose", None)
    try:
        model_class(**params)
        return params
    except Exception as e:
        # FIX B5: was a silent fallback to default parameters
        print(f"[Warning] {model_class.__name__}: tuned parameters rejected ({type(e).__name__}: {e}); "
              "falling back to the model's DEFAULT parameters")
        return {}

def _train_alpha_net(
    base_model,
    X_calib,
    y_calib,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    learning_rate=1e-2,  # FIX A3 (was 1e-3)
    min_steps=2000,  # FIX A3
    alpha_clip=1e-3,
    seed=42,
):
    X_train_alpha = _build_loo_feature_matrix(base_model, X_calib, y_calib)
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []
        for run_idx in range(num_runs):
            local_seed = seed + run_idx
            torch.manual_seed(local_seed)
            np.random.seed(local_seed)
            random.seed(local_seed)
            dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
            loader = DataLoader(dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)
            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=learning_rate)
            all_losses = []
            all_sizes = []
            all_alphas = []
            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []
                alpha_net.train()
                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)
                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached
                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + lambda_reg * alpha_pred).mean()
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))
                all_losses.append(float(np.mean(epoch_losses)))
                all_sizes.append(float(np.mean(epoch_sizes)))
                all_alphas.append(float(np.mean(epoch_alphas)))
            alpha_net.eval()
            lambda_losses.append(np.asarray(all_losses, dtype=np.float32))
            lambda_sizes.append(np.asarray(all_sizes, dtype=np.float32))
            lambda_alphas.append(np.asarray(all_alphas, dtype=np.float32))
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)
        all_results[lambda_reg] = {
            "all_losses": np.vstack(lambda_losses),
            "all_sizes": np.vstack(lambda_sizes),
            "all_alphas": np.vstack(lambda_alphas),
            "models": lambda_models,
        }
    return all_results

def _moving_average(values, window=10):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr
    if window <= 1 or arr.size < window:
        return arr.copy()
    kernel = np.ones(window, dtype=float) / float(window)
    return np.convolve(arr, kernel, mode="valid")

def _smooth_runs(run_matrix, window=10):
    smoothed_runs = [_moving_average(run_curve, window=window) for run_curve in np.asarray(run_matrix)]
    if not smoothed_runs:
        return np.array([]), np.array([])
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.array([]), np.array([])
    stacked = np.asarray([curve[:min_len] for curve in smoothed_runs], dtype=float)
    return stacked.mean(axis=0), stacked.std(axis=0)

def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    colors = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00"]
    for idx, (lambda_reg, result_dict) in enumerate(sorted(all_results.items(), key=lambda item: item[0])):
        color = colors[idx % len(colors)]
        loss_mean, loss_std = _smooth_runs(result_dict["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result_dict["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result_dict["all_alphas"], window=10)
        if loss_mean.size == 0:
            continue
        epochs_axis = np.arange(1, len(loss_mean) + 1)
        axes[0].plot(epochs_axis, loss_mean, color=color, label=f"lambda={lambda_reg}")
        axes[0].fill_between(epochs_axis, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)
        axes[1].plot(epochs_axis, size_mean, color=color, label=f"lambda={lambda_reg}")
        axes[1].fill_between(epochs_axis, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)
        axes[2].plot(epochs_axis, alpha_mean, color=color, label=f"lambda={lambda_reg}")
        axes[2].fill_between(epochs_axis, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)
    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Mean Size")
    axes[2].set_title(r"Smoothed Mean $\tilde{\alpha}$")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel(r"Mean $\tilde{\alpha}$")
    for ax in axes:
        ax.grid(alpha=0.3)
        ax.legend(loc="best", fontsize="small")
    plt.tight_layout()
    return fig

def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    final_losses = all_results[lambda_main]["all_losses"][:, -1]
    run_idx = int(np.argmin(final_losses))
    return lambda_main, run_idx, all_results[lambda_main]["models"][run_idx]


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    calib_scores = np.abs(model.predict(X_calib_np) - y_calib_np)
    test_feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(test_feature), min=1e-3, max=1.0 - 1e-3).item())
    score_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(interval_size(score_tensor, torch.tensor(alpha_hat, device=DEVICE)).item())
    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width
    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))
    return {
        "X_test": X_test_np,
        "y_test": y_test_np,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "alpha": alpha_hat,
        "interval_width": interval_width,
        "coverage": coverage,
        "rmse": rmse,
        "mae": mae,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }

def conformal_predictions_ACP(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    alpha=ALPHA,  # FIX M4: was 0.1 (nominal only; AlphaNet learns alpha~)
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)
    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np,
        y_train_np,
        test_size=0.35,
        random_state=42,
    )
    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)
    all_results = _train_alpha_net(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )
    best_lambda, best_run_idx, alpha_net = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    eval_bundle = evaluate_acp(
        model=model,
        alpha_net=alpha_net,
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )
    y_true = eval_bundle["y_test"]
    y_pred = eval_bundle["y_pred"]
    lower = eval_bundle["lower"]
    upper = eval_bundle["upper"]
    print(f"{model_name} ACP best lambda: {best_lambda}")
    print(f"{model_name} ACP best run: {best_run_idx + 1}")
    print(f"{model_name} ACP alpha: {eval_bundle['alpha']:.4f}")
    print(f"{model_name} ACP width: {eval_bundle['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {eval_bundle['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {eval_bundle['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {eval_bundle['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {eval_bundle['target_coverage'] * 100:.2f}%  |  miss / alpha~: {eval_bundle['miss_over_alpha']:.3f}  # FIX A1")
    fig_pred, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))
    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)
    plt.tight_layout()
    fig_dynamics = plot_acp_training_dynamics(all_results)
    if excel_file_path:
        save_plot_to_excel(fig_pred, excel_file_path, "conformal_predictions_ACP")
        save_plot_to_excel(fig_dynamics, excel_file_path, "acp_training_dynamics")
        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), eval_bundle["alpha"]),
                "ACP_interval_width": np.full(len(y_true), eval_bundle["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_best_lambda": np.full(len(y_true), best_lambda),
                "ACP_best_run": np.full(len(y_true), best_run_idx + 1),
                "ACP_target_coverage": np.full(len(y_true), eval_bundle["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), eval_bundle["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )
        training_summary_rows = []
        training_curve_rows = []
        for lambda_reg, result_dict in sorted(all_results.items(), key=lambda item: item[0]):
            final_losses = result_dict["all_losses"][:, -1]
            final_sizes = result_dict["all_sizes"][:, -1]
            final_alphas = result_dict["all_alphas"][:, -1]
            training_summary_rows.append(
                {
                    "lambda": lambda_reg,
                    "runs": result_dict["all_losses"].shape[0],
                    "best_run": int(np.argmin(final_losses)) + 1,
                    "final_loss_mean": float(np.mean(final_losses)),
                    "final_loss_std": float(np.std(final_losses)),
                    "final_size_mean": float(np.mean(final_sizes)),
                    "final_size_std": float(np.std(final_sizes)),
                    "final_alpha_mean": float(np.mean(final_alphas)),
                    "final_alpha_std": float(np.std(final_alphas)),
                }
            )
            mean_loss_curve = result_dict["all_losses"].mean(axis=0)
            mean_size_curve = result_dict["all_sizes"].mean(axis=0)
            mean_alpha_curve = result_dict["all_alphas"].mean(axis=0)
            for epoch_idx in range(len(mean_loss_curve)):
                training_curve_rows.append(
                    {
                        "lambda": lambda_reg,
                        "epoch": epoch_idx + 1,
                        "mean_loss": float(mean_loss_curve[epoch_idx]),
                        "mean_size": float(mean_size_curve[epoch_idx]),
                        "mean_alpha": float(mean_alpha_curve[epoch_idx]),
                    }
                )
        save_dataframe_to_excel(pd.DataFrame(training_summary_rows), excel_file_path, "acp_training_summary")
        save_dataframe_to_excel(pd.DataFrame(training_curve_rows), excel_file_path, "acp_training_curves")
        save_dataframe_to_excel(_acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np),
                                excel_file_path, "acp_lambda_comparison")  # FIX A1
    plt.close(fig_pred)
    plt.close(fig_dynamics)
    eval_bundle["model"] = model
    eval_bundle["alpha_net"] = alpha_net
    eval_bundle["all_results"] = all_results
    eval_bundle["best_lambda"] = best_lambda
    eval_bundle["best_run"] = best_run_idx + 1
    return eval_bundle


In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

def _train_alpha_net_multi(
    base_model,
    X_calib,
    y_calib,
    X_train_alpha,
    lambdas=(10, 20, 50),
    num_runs=3,
    epochs=50,
    batch_size=32,
    lr=1e-2,  # FIX A3 (was 1e-3)
    alpha_clip=1e-3,
    min_steps=2000,  # FIX A3
    seed=42,
):
    # FIX A2: the network input (sum of leave-one-out errors) is divided by its mean, so inputs are ~1 on
    # every dataset (raw sums can be in the thousands, where the sigmoid is flat and training stalls).
    # The same number divides the test feature in evaluate_acp (stored on each net as feature_scale_).
    feature_scale = float(X_train_alpha.mean())
    feature_scale = feature_scale if feature_scale > 0 else 1.0
    X_train_alpha = X_train_alpha / feature_scale
    # FIX A3: leave-one-out score vectors computed once (identical values; was one predict() per row per step)
    loo_scores = [torch.tensor(_compute_scores_without_index(base_model, X_calib, y_calib, i), dtype=torch.float32, device=DEVICE)
                  for i in range(len(X_calib))]
    Dataset = TensorDataset(X_train_alpha, torch.arange(len(X_train_alpha), dtype=torch.long))
    loader = DataLoader(Dataset, batch_size=min(batch_size, len(X_train_alpha)), shuffle=True)

    all_results = {}
    for lambda_reg in lambdas:
        lambda_losses = []
        lambda_sizes = []
        lambda_alphas = []
        lambda_models = []

        for run_idx in range(num_runs):
            torch.manual_seed(seed + run_idx)
            np.random.seed(seed + run_idx)

            alpha_net = AlphaNet(input_dim=X_train_alpha.shape[1]).to(DEVICE)
            optimizer = optim.Adam(alpha_net.parameters(), lr=lr)

            run_losses = []
            run_sizes = []
            run_alphas = []

            # FIX A3: AlphaNet trained for at least 2000 optimizer steps at lr 1e-2 (was 50 epochs at lr 1e-3).
            # With a small calibration set one epoch is one step, so 50 steps left alpha~ at its start
            # value (~0.98) for every lambda. Checked on pile (run REG22_11_2026_08_09): the trained alpha~
            # now matches the exact minimiser of the same loss for every lambda.
            n_epochs = max(epochs, int(np.ceil(min_steps / len(loader))))
            for _ in range(n_epochs):
                alpha_net.train()
                epoch_losses = []
                epoch_sizes = []
                epoch_alphas = []

                for x_batch, idx_batch in loader:
                    x_batch = x_batch.to(DEVICE)
                    alpha_pred = torch.clamp(alpha_net(x_batch), min=alpha_clip, max=1.0 - alpha_clip)

                    batch_sizes = []
                    for j, idx in enumerate(idx_batch.tolist()):
                        batch_sizes.append(interval_size(loo_scores[idx], alpha_pred[j]))  # FIX A3: cached

                    batch_sizes = torch.stack(batch_sizes)
                    loss = (batch_sizes + float(lambda_reg) * alpha_pred).mean()

                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

                    epoch_losses.append(float(loss.item()))
                    epoch_sizes.append(float(batch_sizes.mean().item()))
                    epoch_alphas.append(float(alpha_pred.mean().item()))

                run_losses.append(float(np.mean(epoch_losses)))
                run_sizes.append(float(np.mean(epoch_sizes)))
                run_alphas.append(float(np.mean(epoch_alphas)))

            alpha_net.eval()
            lambda_losses.append(run_losses)
            lambda_sizes.append(run_sizes)
            lambda_alphas.append(run_alphas)
            alpha_net.feature_scale_ = feature_scale  # FIX A2
            lambda_models.append(alpha_net)

        all_results[float(lambda_reg)] = {
            "all_losses": np.asarray(lambda_losses, dtype=np.float32),
            "all_sizes": np.asarray(lambda_sizes, dtype=np.float32),
            "all_alphas": np.asarray(lambda_alphas, dtype=np.float32),
            "models": lambda_models,
        }

    return all_results


def _moving_average(values, window=10):
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0:
        return values
    window = max(1, min(int(window), values.size))
    kernel = np.ones(window, dtype=np.float32) / window
    return np.convolve(values, kernel, mode="valid")


def _smooth_runs(run_matrix, window=10):
    run_matrix = np.asarray(run_matrix, dtype=np.float32)
    if run_matrix.ndim == 1:
        run_matrix = run_matrix.reshape(1, -1)
    if run_matrix.size == 0:
        return np.asarray([]), np.asarray([])

    smoothed_runs = [_moving_average(run, window=window) for run in run_matrix]
    min_len = min(len(curve) for curve in smoothed_runs)
    if min_len == 0:
        return np.asarray([]), np.asarray([])

    trimmed = np.vstack([curve[:min_len] for curve in smoothed_runs])
    return trimmed.mean(axis=0), trimmed.std(axis=0)


def plot_acp_training_dynamics(all_results):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    if not all_results:
        for ax in axes:
            ax.text(0.5, 0.5, "No training results", ha="center", va="center")
            ax.axis("off")
        return fig

    sorted_results = sorted(all_results.items(), key=lambda item: item[0])
    colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(sorted_results)))

    for color, (lambda_reg, result) in zip(colors, sorted_results):
        loss_mean, loss_std = _smooth_runs(result["all_losses"], window=10)
        size_mean, size_std = _smooth_runs(result["all_sizes"], window=10)
        alpha_mean, alpha_std = _smooth_runs(result["all_alphas"], window=10)

        if loss_mean.size:
            x_loss = np.arange(1, loss_mean.size + 1)
            axes[0].plot(x_loss, loss_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[0].fill_between(x_loss, loss_mean - loss_std, loss_mean + loss_std, color=color, alpha=0.2)

        if size_mean.size:
            x_size = np.arange(1, size_mean.size + 1)
            axes[1].plot(x_size, size_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[1].fill_between(x_size, size_mean - size_std, size_mean + size_std, color=color, alpha=0.2)

        if alpha_mean.size:
            x_alpha = np.arange(1, alpha_mean.size + 1)
            axes[2].plot(x_alpha, alpha_mean, color=color, label=f"lambda={lambda_reg:g}")
            axes[2].fill_between(x_alpha, alpha_mean - alpha_std, alpha_mean + alpha_std, color=color, alpha=0.2)

    axes[0].set_title("Smoothed Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")

    axes[1].set_title("Smoothed Mean Size")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Size")

    axes[2].set_title("Smoothed Mean Alpha-tilde")
    axes[2].set_xlabel("Epoch")
    axes[2].set_ylabel("Alpha-tilde")

    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(loc="best", fontsize="small")

    plt.tight_layout()
    return fig


def _select_best_alpha_net(all_results, lambda_main=50):
    # FIX A1: lambda is the user's size/coverage trade-off; the best RUN is chosen within lambda_main by its
    # final training loss. Was: final losses compared ACROSS lambdas; since the loss contains lambda * alpha~,
    # the smallest lambda (10, the narrowest / lowest-coverage setting) always won.
    if lambda_main not in all_results:
        raise ValueError(f"lambda_main={lambda_main} not in trained lambdas {sorted(all_results)}")
    losses = all_results[lambda_main]["all_losses"]
    if losses.size == 0:
        raise RuntimeError("AlphaNet training produced no valid runs.")
    final_losses = losses[:, -1]
    run_idx = int(np.argmin(final_losses))
    return {
        "lambda": lambda_main,
        "run_idx": run_idx,
        "final_loss": float(final_losses[run_idx]),
        "model": all_results[lambda_main]["models"][run_idx],
    }


def evaluate_acp(model, alpha_net, X_calib, y_calib, X_test, y_test):
    X_calib_np = _to_numpy_2d(X_calib)
    y_calib_np = _to_numpy_1d(y_calib)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    calib_preds = np.asarray(model.predict(X_calib_np)).reshape(-1)
    calib_scores = np.abs(calib_preds - y_calib_np)

    feature = torch.tensor([[calib_scores.sum() / getattr(alpha_net, "feature_scale_", 1.0)]], dtype=torch.float32, device=DEVICE)  # FIX A2
    with torch.no_grad():
        alpha_hat = float(torch.clamp(alpha_net(feature), min=1e-3, max=1.0 - 1e-3).item())

    scores_tensor = torch.tensor(calib_scores, dtype=torch.float32, device=DEVICE)
    interval_width = float(
        interval_size(scores_tensor, torch.tensor(alpha_hat, dtype=torch.float32, device=DEVICE)).item()
    )

    y_pred = np.asarray(model.predict(X_test_np)).reshape(-1)
    lower = y_pred - 0.5 * interval_width
    upper = y_pred + 0.5 * interval_width

    coverage = float(np.mean((y_test_np >= lower) & (y_test_np <= upper)))
    rmse = float(np.sqrt(mean_squared_error(y_test_np, y_pred)))
    mae = float(np.mean(np.abs(y_test_np - y_pred)))

    return {
        "coverage": coverage,
        "interval_width": interval_width,
        "alpha_hat": alpha_hat,
        "rmse": rmse,
        "mae": mae,
        "y_pred": y_pred,
        "lower": lower,
        "upper": upper,
        "y_test": y_test_np,
        # FIX A1: the guarantee is about the LEARNED level (eq. 3: E[miss / alpha~] <= 1), so report
        # coverage against 1 - alpha~ and the miss rate divided by alpha~.
        "target_coverage": 1.0 - alpha_hat,
        "miss_over_alpha": (1.0 - coverage) / alpha_hat,
    }


def run_full_acp_pipeline(
    model_class,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name,
    excel_file_path,
    lambdas=(10, 20, 50),
    lambda_main=50,  # FIX A1: the lambda used for the main result (paper reports lambda = 50)
    num_runs=3,
    epochs=50,
):
    X_train_np = _to_numpy_2d(X_train)
    y_train_np = _to_numpy_1d(y_train)
    X_test_np = _to_numpy_2d(X_test)
    y_test_np = _to_numpy_1d(y_test)

    X_fit, X_calib, y_fit, y_calib = train_test_split(
        X_train_np, y_train_np, test_size=0.35, random_state=42
    )

    safe_params = _resolve_model_params(model_class, best_params)
    model = model_class(**safe_params)
    model.fit(X_fit, y_fit)

    X_train_alpha = _build_loo_feature_matrix(model, X_calib, y_calib)
    all_results = _train_alpha_net_multi(
        base_model=model,
        X_calib=X_calib,
        y_calib=y_calib,
        X_train_alpha=X_train_alpha,
        lambdas=lambdas,
        num_runs=num_runs,
        epochs=epochs,
    )

    dynamics_fig = plot_acp_training_dynamics(all_results)
    dynamics_fig.suptitle(f"ACP Training Dynamics - {model_name}")
    dynamics_fig.tight_layout(rect=[0, 0, 1, 0.95])
    if excel_file_path:
        save_plot_to_excel(dynamics_fig, excel_file_path, "acp_training_dynamics")
    plt.close(dynamics_fig)

    best_choice = _select_best_alpha_net(all_results, lambda_main)  # FIX A1
    metrics = evaluate_acp(
        model=model,
        alpha_net=best_choice["model"],
        X_calib=X_calib,
        y_calib=y_calib,
        X_test=X_test_np,
        y_test=y_test_np,
    )

    y_true = metrics["y_test"]
    y_pred = metrics["y_pred"]
    lower = metrics["lower"]
    upper = metrics["upper"]

    print(f"{model_name} best lambda: {best_choice['lambda']}")
    print(f"{model_name} best run: {best_choice['run_idx']}")
    print(f"{model_name} ACP alpha_hat: {metrics['alpha_hat']:.4f}")
    print(f"{model_name} ACP width: {metrics['interval_width']:.4f}")
    print(f"{model_name} ACP coverage: {metrics['coverage'] * 100:.2f}%")
    print(f"{model_name} ACP RMSE: {metrics['rmse']:.4f}")
    print(f"{model_name} ACP MAE: {metrics['mae']:.4f}")
    print(f"{model_name} ACP target coverage (1 - alpha~): {metrics['target_coverage'] * 100:.2f}%  |  miss / alpha~: {metrics['miss_over_alpha']:.3f}  # FIX A1")

    fig, axs = plt.subplots(2, 1, figsize=(11, 12))
    idx = np.arange(len(y_true))

    axs[0].scatter(idx, y_true, label="True", color="blue", s=12, alpha=0.7)
    axs[0].fill_between(idx, lower, upper, color="gray", alpha=0.45, label="ACP Interval")
    axs[0].plot(idx, y_pred, color="red", linewidth=1.2, label="Prediction")
    axs[0].set_title(f"ACP Intervals - {model_name}")
    axs[0].set_xlabel("Sample Number")
    axs[0].set_ylabel(f"{TARGET_LABEL}")
    axs[0].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    abs_err = np.abs(y_true - y_pred)
    widths = upper - lower
    axs[1].scatter(widths, abs_err, s=14, alpha=0.7, color="teal", label="Samples")
    axs[1].axvline(np.mean(widths), linestyle="--", color="gray", label="Mean Width")
    axs[1].axhline(np.mean(abs_err), linestyle="--", color="orange", label="Mean Absolute Error")
    axs[1].set_title(f"ACP Width vs Absolute Error - {model_name}")
    axs[1].set_xlabel("Interval Width")
    axs[1].set_ylabel("Absolute Error")
    axs[1].legend(loc="upper left", fontsize="small", bbox_to_anchor=(1.02, 1), borderaxespad=0.0)

    plt.tight_layout()

    if excel_file_path:
        save_plot_to_excel(fig, excel_file_path, "conformal_predictions_ACP")

        save_values_to_excel({
                "ACP_true": y_true,
                "ACP_pred": y_pred,
                "ACP_lower": lower,
                "ACP_upper": upper,
                "ACP_alpha": np.full(len(y_true), metrics["alpha_hat"]),
                "ACP_interval_width": np.full(len(y_true), metrics["interval_width"]),
                "ACP_is_covered": ((y_true >= lower) & (y_true <= upper)).astype(int),
                "ACP_target_coverage": np.full(len(y_true), metrics["target_coverage"]),  # FIX A1
                "ACP_miss_over_alpha": np.full(len(y_true), metrics["miss_over_alpha"]),  # FIX A1
            },
            excel_file_path,
            "conformal_predictions_ACP_values",
        )

        save_values_to_excel({
                "model_name": [model_name],
                "best_lambda": [best_choice["lambda"]],
                "best_run": [best_choice["run_idx"]],
                "best_final_loss": [best_choice["final_loss"]],
                "coverage": [metrics["coverage"]],
                "interval_width": [metrics["interval_width"]],
                "rmse": [metrics["rmse"]],
                "mae": [metrics["mae"]],
                "alpha_hat": [metrics["alpha_hat"]],
                "Target_coverage": [metrics["target_coverage"]],  # FIX A1
                "Miss_over_alpha": [metrics["miss_over_alpha"]],  # FIX A1
            },
            excel_file_path,
            "acp_final_metrics",
        )

        summary = {
            "lambda": [],
            "mean_final_loss": [],
            "std_final_loss": [],
            "mean_final_size": [],
            "std_final_size": [],
            "mean_final_alpha": [],
            "std_final_alpha": [],
        }
        curve_data = {
            "lambda": [],
            "run": [],
            "epoch": [],
            "loss": [],
            "size": [],
            "alpha_tilde": [],
        }

        for lambda_reg, result in sorted(all_results.items(), key=lambda item: item[0]):
            losses = result["all_losses"]
            sizes = result["all_sizes"]
            alphas = result["all_alphas"]

            summary["lambda"].append(lambda_reg)
            summary["mean_final_loss"].append(float(losses[:, -1].mean()))
            summary["std_final_loss"].append(float(losses[:, -1].std()))
            summary["mean_final_size"].append(float(sizes[:, -1].mean()))
            summary["std_final_size"].append(float(sizes[:, -1].std()))
            summary["mean_final_alpha"].append(float(alphas[:, -1].mean()))
            summary["std_final_alpha"].append(float(alphas[:, -1].std()))

            n_runs, n_epochs = losses.shape
            for run_idx in range(n_runs):
                for epoch_idx in range(n_epochs):
                    curve_data["lambda"].append(lambda_reg)
                    curve_data["run"].append(run_idx)
                    curve_data["epoch"].append(epoch_idx + 1)
                    curve_data["loss"].append(float(losses[run_idx, epoch_idx]))
                    curve_data["size"].append(float(sizes[run_idx, epoch_idx]))
                    curve_data["alpha_tilde"].append(float(alphas[run_idx, epoch_idx]))

        save_values_to_excel(summary, excel_file_path, "acp_training_summary")
        save_values_to_excel(curve_data, excel_file_path, "acp_training_curves")
        lambda_df = _acp_lambda_comparison(all_results, model, X_calib, y_calib, X_test_np, y_test_np)  # FIX A1
        print(lambda_df.to_string(index=False))
        save_dataframe_to_excel(lambda_df, excel_file_path, "acp_lambda_comparison")

    plt.close(fig)

    return {
        "model": model,
        "all_results": all_results,
        "best_choice": best_choice,
        "metrics": metrics,
    }


In [15]:
# Unified ACP execution entry point:
# run_full_acp_pipeline(model_class, best_params, X_train, y_train, X_test, y_test, model_name, excel_file_path)


# **Tuned Parameters**


In [16]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/scour_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/scour-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


# **Conformal Predictions with Lightgbm**


In [17]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')


In [18]:
run_full_acp_pipeline(
    LGBMRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "LightGBM",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/LightGBM.xlsx",
)


LightGBM best lambda: 50
LightGBM best run: 1
LightGBM ACP alpha_hat: 0.1331
LightGBM ACP width: 6.0047
LightGBM ACP coverage: 100.00%
LightGBM ACP RMSE: 0.4178
LightGBM ACP MAE: 0.3001
LightGBM ACP target coverage (1 - alpha~): 86.69%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          5.439696       0.278844         0.721156            1.0 2.647096              0.0
   20.0                   2          7.803102       0.201831         0.798169            1.0 3.757143              0.0
   50.0                   2         12.683210       0.133091         0.866909            1.0 6.004722              0.0


{'model': LGBMRegressor(bagging_freq=1, learning_rate=0.05, max_depth=3,
               min_child_samples=10, min_child_weight=0.01, n_estimators=500,
               n_jobs=-1, num_leaves=63, objective='regression', random_state=42,
               reg_alpha=0.01, reg_lambda=10, verbose=-1),
 'all_results': {10.0: {'all_losses': array([[10.640131 , 10.614155 , 10.574624 , ...,  5.440423 ,  5.4389987,
            5.4422083],
          [10.638398 , 10.62121  , 10.598536 , ...,  5.4368086,  5.4412956,
            5.4396963],
          [10.635526 , 10.615677 , 10.586847 , ...,  5.4417458,  5.442401 ,
            5.4402113]], dtype=float32),
   'all_sizes': array([[0.70790684, 0.7099227 , 0.7122834 , ..., 2.6063619 , 2.607433  ,
           2.667346  ],
          [0.7077112 , 0.70864826, 0.71055233, ..., 2.6277232 , 2.6473947 ,
           2.6136713 ],
          [0.7080029 , 0.70970786, 0.71163666, ..., 2.5931058 , 2.6508262 ,
           2.6259956 ]], dtype=float32),
   'all_alphas': array([[0

# **Conformal Predictions with XGBoost**


In [19]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')


In [20]:
run_full_acp_pipeline(
    XGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "XGBoost",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/XGBoost.xlsx",
)


XGBoost best lambda: 50
XGBoost best run: 1
XGBoost ACP alpha_hat: 0.1216
XGBoost ACP width: 5.4287
XGBoost ACP coverage: 100.00%
XGBoost ACP RMSE: 0.4106
XGBoost ACP MAE: 0.2927
XGBoost ACP target coverage (1 - alpha~): 87.84%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          4.926690       0.254360         0.745640       0.987179 2.377981         0.050403
   20.0                   2          7.077569       0.184215         0.815785       1.000000 3.382617         0.000000
   50.0                   2         11.536358       0.121638         0.878362       1.000000 5.428671         0.000000


{'model': XGBRegressor(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=0.5, colsample_bynode=None, colsample_bytree=0.9,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=0, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.15, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=-1,
              num_parallel_tree=None, ...),
 'all_results': {10.0: {'all_losses': array([[10.508371 , 10.482034 , 10.4420395, ...,  4.927042 ,  4.9248877,
            4.9292717],
          [10.506814 , 10.489418 , 10.466276 , ...,  4.9233494,  4.927474 ,
            4.9266896],
        

# **Conformal Predictions with GPBoost**


In [21]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')


In [22]:
run_full_acp_pipeline(
    GPBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GPBoost",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/GPBoost.xlsx",
)


GPBoost best lambda: 50
GPBoost best run: 1
GPBoost ACP alpha_hat: 0.1281
GPBoost ACP width: 5.7172
GPBoost ACP coverage: 100.00%
GPBoost ACP RMSE: 0.4127
GPBoost ACP MAE: 0.2952
GPBoost ACP target coverage (1 - alpha~): 87.19%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          5.199817       0.267934         0.732066            1.0 2.515821              0.0
   20.0                   2          7.463857       0.193879         0.806121            1.0 3.576215              0.0
   50.0                   2         12.147113       0.128084         0.871916            1.0 5.717206              0.0


{'model': GPBoostRegressor(learning_rate=0.05, max_depth=3, min_child_samples=10,
                  min_child_weight=0.1, n_estimators=500, num_leaves=15,
                  random_state=42, reg_alpha=1.0, reg_lambda=1.0, subsample=0.8,
                  verbose=-1),
 'all_results': {10.0: {'all_losses': array([[10.576931 , 10.550838 , 10.510994 , ...,  5.2003145,  5.1985536,
            5.202025 ],
          [10.575087 , 10.557922 , 10.534933 , ...,  5.1966066,  5.2016335,
            5.1998167],
          [10.5722885, 10.552172 , 10.523183 , ...,  5.201886 ,  5.2021694,
            5.1999607]], dtype=float32),
   'all_sizes': array([[0.64470863, 0.64661646, 0.64865804, ..., 2.4979029 , 2.4778304 ,
           2.5441914 ],
          [0.6443978 , 0.6453619 , 0.64694184, ..., 2.5052173 , 2.5327    ,
           2.4926906 ],
          [0.64476657, 0.6461962 , 0.6479639 , ..., 2.4725108 , 2.5316005 ,
           2.5058174 ]], dtype=float32),
   'all_alphas': array([[0.9932221 , 0.99042225, 0.

# **Conformal Predictions with NGBoost**


In [23]:
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')


In [24]:
run_full_acp_pipeline(
    NGBRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "NGBoost",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/NGBoost.xlsx",
)


NGBoost best lambda: 50
NGBoost best run: 1
NGBoost ACP alpha_hat: 0.1251
NGBoost ACP width: 5.5314
NGBoost ACP coverage: 100.00%
NGBoost ACP RMSE: 0.3790
NGBoost ACP MAE: 0.2735
NGBoost ACP target coverage (1 - alpha~): 87.49%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          5.049399       0.260803         0.739197            1.0 2.436943              0.0
   20.0                   2          7.251220       0.188782         0.811218            1.0 3.465733              0.0
   50.0                   2         11.811323       0.125073         0.874927            1.0 5.531358              0.0


{'model': NGBRegressor(learning_rate=0.03, minibatch_frac=0.5, n_estimators=200,
              random_state=RandomState(MT19937) at 0x7FE1B3256B40, verbose=0),
 'all_results': {10.0: {'all_losses': array([[10.5388775, 10.512676 , 10.472739 , ...,  5.050867 ,  5.048298 ,
            5.0524664],
          [10.537312 , 10.519606 , 10.496545 , ...,  5.045418 ,  5.0501747,
            5.0493994],
          [10.534197 , 10.514284 , 10.484933 , ...,  5.0513134,  5.0518064,
            5.0499673]], dtype=float32),
   'all_sizes': array([[0.60665584, 0.60845375, 0.61040884, ..., 2.4187174 , 2.4072425 ,
           2.4690208 ],
          [0.6066277 , 0.6070343 , 0.60853434, ..., 2.4293866 , 2.4524167 ,
           2.423514  ],
          [0.60667086, 0.6083307 , 0.60971403, ..., 2.3984938 , 2.4628682 ,
           2.428075  ]], dtype=float32),
   'all_alphas': array([[0.9932222 , 0.99042225, 0.986233  , ..., 0.263215  , 0.26410556,
           0.25834453],
          [0.9930684 , 0.9912571 , 0.988801 

# **Conformal Predictions with Gradient Boosting**


In [25]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')


In [26]:
run_full_acp_pipeline(
    GradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "GradientBoosting",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/Gradient Boosting.xlsx",
)


GradientBoosting best lambda: 50
GradientBoosting best run: 1
GradientBoosting ACP alpha_hat: 0.1314
GradientBoosting ACP width: 5.9519
GradientBoosting ACP coverage: 100.00%
GradientBoosting ACP RMSE: 0.3864
GradientBoosting ACP MAE: 0.2852
GradientBoosting ACP target coverage (1 - alpha~): 86.86%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          5.377252       0.275121         0.724879            1.0 2.621705              0.0
   20.0                   2          7.714704       0.198749         0.801251            1.0 3.730573              0.0
   50.0                   2         12.543055       0.131358         0.868642            1.0 5.951943              0.0


{'model': GradientBoostingRegressor(max_depth=10, max_features='log2',
                           min_impurity_decrease=0.1, min_samples_leaf=5,
                           min_samples_split=10, min_weight_fraction_leaf=0.01,
                           random_state=42, subsample=0.5, tol=0.001),
 'all_results': {10.0: {'all_losses': array([[10.62316  , 10.597366 , 10.55784  , ...,  5.3775167,  5.3756647,
            5.3796177],
          [10.621544 , 10.604383 , 10.58156  , ...,  5.37474  ,  5.3783484,
            5.377252 ],
          [10.618756 , 10.599028 , 10.570009 , ...,  5.3797045,  5.380211 ,
            5.3778195]], dtype=float32),
   'all_sizes': array([[0.6909333 , 0.6931385 , 0.69552076, ..., 2.5826893 , 2.566836  ,
           2.6393595 ],
          [0.69085574, 0.6918212 , 0.6935681 , ..., 2.590304  , 2.6152966 ,
           2.5889688 ],
          [0.6912348 , 0.6930719 , 0.69480485, ..., 2.5490327 , 2.6203897 ,
           2.6045597 ]], dtype=float32),
   'all_alphas': array

# **Conformal Predictions with CatBoost**


In [27]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')


In [28]:
run_full_acp_pipeline(
    CatBoostRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "CatBoost",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/CatBoost.xlsx",
)


CatBoost best lambda: 50
CatBoost best run: 1
CatBoost ACP alpha_hat: 0.1249
CatBoost ACP width: 5.5943
CatBoost ACP coverage: 100.00%
CatBoost ACP RMSE: 0.3397
CatBoost ACP MAE: 0.2501
CatBoost ACP target coverage (1 - alpha~): 87.51%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          5.073587       0.261169         0.738831            1.0 2.457130              0.0
   20.0                   2          7.285279       0.189070         0.810930            1.0 3.493811              0.0
   50.0                   2         11.864565       0.124907         0.875093            1.0 5.594278              0.0


{'model': <catboost.core.CatBoostRegressor at 0x7fe1b33b7ed0>,
 'all_results': {10.0: {'all_losses': array([[10.544952 , 10.518635 , 10.47868  , ...,  5.0744257,  5.07234  ,
            5.0768456],
          [10.543173 , 10.525633 , 10.502658 , ...,  5.0703144,  5.074323 ,
            5.0735874],
          [10.540163 , 10.520285 , 10.49105  , ...,  5.0753894,  5.0747643,
            5.074519 ]], dtype=float32),
   'all_sizes': array([[0.61273384, 0.61441207, 0.61633885, ..., 2.4331372 , 2.4107976 ,
           2.488569  ],
          [0.6124881 , 0.61306524, 0.61466134, ..., 2.4429295 , 2.4672189 ,
           2.4330277 ],
          [0.61263955, 0.61433303, 0.6158542 , ..., 2.4013422 , 2.4624104 ,
           2.4615602 ]], dtype=float32),
   'all_alphas': array([[0.9932219 , 0.99042225, 0.9862342 , ..., 0.2641288 , 0.26615426,
           0.25882766],
          [0.99306846, 0.9912568 , 0.9887998 , ..., 0.26273847, 0.26071042,
           0.264056  ],
          [0.9927524 , 0.9905951 , 0.9875

# **Conformal Predictions with HistGradientBoosting**


In [29]:
best_params = get_best_model_params(best_scores_autosampler, 'HGBR')


In [30]:
run_full_acp_pipeline(
    HistGradientBoostingRegressor,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "HGBR",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/HGBM.xlsx",
)


HGBR best lambda: 50
HGBR best run: 1
HGBR ACP alpha_hat: 0.1240
HGBR ACP width: 5.4879
HGBR ACP coverage: 100.00%
HGBR ACP RMSE: 0.3985
HGBR ACP MAE: 0.2870
HGBR ACP target coverage (1 - alpha~): 87.60%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   3          5.005362       0.257129         0.742871       0.987179 2.429459          0.04986
   20.0                   2          7.189116       0.187166         0.812834       1.000000 3.435303          0.00000
   50.0                   2         11.712723       0.123962         0.876038       1.000000 5.487946          0.00000


{'model': HistGradientBoostingRegressor(early_stopping=False, l2_regularization=1.0,
                               learning_rate=0.05, max_depth=7,
                               min_samples_leaf=5, random_state=42),
 'all_results': {10.0: {'all_losses': array([[10.527721 , 10.501529 , 10.4616   , ...,  5.006737 ,  5.0042896,
            5.0071845],
          [10.526072 , 10.508629 , 10.485481 , ...,  5.001562 ,  5.0070353,
            5.0055513],
          [10.523099 , 10.503168 , 10.4738865, ...,  5.007303 ,  5.006996 ,
            5.0053616]], dtype=float32),
   'all_sizes': array([[0.595497  , 0.5972986 , 0.59925807, ..., 2.3928924 , 2.3877969 ,
           2.443003  ],
          [0.59538496, 0.59606093, 0.5974699 , ..., 2.40269   , 2.438665  ,
           2.4003267 ],
          [0.5955732 , 0.59720904, 0.59867144, ..., 2.372665  , 2.4271388 ,
           2.41685   ]], dtype=float32),
   'all_alphas': array([[0.99322253, 0.990423  , 0.9862342 , ..., 0.26138455, 0.26164928,
          

# **Conformal Predictions with PGBM**


In [31]:
class PGBMWrapper(BaseEstimator, RegressorMixin):
    def __init__(self, **params):
        self.params = params
        self.model = None

    def fit(self, X, y):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        y_ = y.to_numpy() if hasattr(y, "to_numpy") else np.array(y)
        self.model = PGBM()
        self.model.train(
            train_set=(X_, y_),
            objective=mseloss_objective,
            metric=rmseloss_metric,
            params=self.params
        )
        return self

    def predict(self, X):
        X_ = X.to_numpy() if hasattr(X, "to_numpy") else np.array(X)
        return self.model.predict(X_).numpy()


In [32]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()
def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params=best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Define the quantiles you want to predict
quantiles = [0.05, 0.1, 0.15, 0.2, 0.3,
             0.4, 0.5, 0.6, 0.7, 0.8,
             0.85, 0.9, 0.95]

# DataFrame to store predictions
predictions_PGBM_df = pd.DataFrame()

# Calculate and store quantiles
for q in quantiles:
    predictions_PGBM_df[q] = np.quantile(pred_dist, q, axis=0)

# Add actual target values to the DataFrame
predictions_PGBM_df['Actual'] = y_test.ravel()

# Print the DataFrame with predictions for each quantile
print(predictions_PGBM_df.head())


Training on CPU


Estimator 0/300, Train metric: 1.2173
Estimator 1/300, Train metric: 1.1687
Estimator 2/300, Train metric: 1.1274
Estimator 3/300, Train metric: 1.0832
Estimator 4/300, Train metric: 1.0460
Estimator 5/300, Train metric: 1.0061
Estimator 6/300, Train metric: 0.9684
Estimator 7/300, Train metric: 0.9329
Estimator 8/300, Train metric: 0.8993
Estimator 9/300, Train metric: 0.8678
Estimator 10/300, Train metric: 0.8406
Estimator 11/300, Train metric: 0.8139
Estimator 12/300, Train metric: 0.7866
Estimator 13/300, Train metric: 0.7607


Estimator 14/300, Train metric: 0.7364
Estimator 15/300, Train metric: 0.7125
Estimator 16/300, Train metric: 0.6900
Estimator 17/300, Train metric: 0.6688
Estimator 18/300, Train metric: 0.6486
Estimator 19/300, Train metric: 0.6299
Estimator 20/300, Train metric: 0.6117
Estimator 21/300, Train metric: 0.5945
Estimator 22/300, Train metric: 0.5781
Estimator 23/300, Train metric: 0.5627
Estimator 24/300, Train metric: 0.5481
Estimator 25/300, Train metric: 0.5346
Estimator 26/300, Train metric: 0.5216
Estimator 27/300, Train metric: 0.5096
Estimator 28/300, Train metric: 0.4972
Estimator 29/300, Train metric: 0.4855
Estimator 30/300, Train metric: 0.4740
Estimator 31/300, Train metric: 0.4631
Estimator 32/300, Train metric: 0.4516
Estimator 33/300, Train metric: 0.4414


Estimator 34/300, Train metric: 0.4316
Estimator 35/300, Train metric: 0.4224
Estimator 36/300, Train metric: 0.4135
Estimator 37/300, Train metric: 0.4051
Estimator 38/300, Train metric: 0.3965
Estimator 39/300, Train metric: 0.3891
Estimator 40/300, Train metric: 0.3821
Estimator 41/300, Train metric: 0.3748
Estimator 42/300, Train metric: 0.3688
Estimator 43/300, Train metric: 0.3631
Estimator 44/300, Train metric: 0.3576
Estimator 45/300, Train metric: 0.3519
Estimator 46/300, Train metric: 0.3466
Estimator 47/300, Train metric: 0.3418
Estimator 48/300, Train metric: 0.3373
Estimator 49/300, Train metric: 0.3324
Estimator 50/300, Train metric: 0.3286
Estimator 51/300, Train metric: 0.3249
Estimator 52/300, Train metric: 0.3215
Estimator 53/300, Train metric: 0.3182
Estimator 54/300, Train metric: 0.3139
Estimator 55/300, Train metric: 0.3109
Estimator 56/300, Train metric: 0.3082
Estimator 57/300, Train metric: 0.3057
Estimator 58/300, Train metric: 0.3017
Estimator 59/300, Train m

Estimator 64/300, Train metric: 0.2892
Estimator 65/300, Train metric: 0.2874
Estimator 66/300, Train metric: 0.2858
Estimator 67/300, Train metric: 0.2848
Estimator 68/300, Train metric: 0.2824
Estimator 69/300, Train metric: 0.2810
Estimator 70/300, Train metric: 0.2773
Estimator 71/300, Train metric: 0.2761
Estimator 72/300, Train metric: 0.2739
Estimator 73/300, Train metric: 0.2719
Estimator 74/300, Train metric: 0.2685
Estimator 75/300, Train metric: 0.2675
Estimator 76/300, Train metric: 0.2644
Estimator 77/300, Train metric: 0.2634
Estimator 78/300, Train metric: 0.2607
Estimator 79/300, Train metric: 0.2576
Estimator 80/300, Train metric: 0.2551
Estimator 81/300, Train metric: 0.2548
Estimator 82/300, Train metric: 0.2533
Estimator 83/300, Train metric: 0.2531
Estimator 84/300, Train metric: 0.2520
Estimator 85/300, Train metric: 0.2512
Estimator 86/300, Train metric: 0.2488
Estimator 87/300, Train metric: 0.2476
Estimator 88/300, Train metric: 0.2474
Estimator 89/300, Train m

Estimator 118/300, Train metric: 0.2394
Estimator 119/300, Train metric: 0.2394
Estimator 120/300, Train metric: 0.2394
Estimator 121/300, Train metric: 0.2393
Estimator 122/300, Train metric: 0.2393
Estimator 123/300, Train metric: 0.2393
Estimator 124/300, Train metric: 0.2393
Estimator 125/300, Train metric: 0.2393
Estimator 126/300, Train metric: 0.2393
Estimator 127/300, Train metric: 0.2393
Estimator 128/300, Train metric: 0.2393
Estimator 129/300, Train metric: 0.2393
Estimator 130/300, Train metric: 0.2393
Estimator 131/300, Train metric: 0.2393
Estimator 132/300, Train metric: 0.2393
Estimator 133/300, Train metric: 0.2393
Estimator 134/300, Train metric: 0.2393
Estimator 135/300, Train metric: 0.2393
Estimator 136/300, Train metric: 0.2393
Estimator 137/300, Train metric: 0.2393
Estimator 138/300, Train metric: 0.2393
Estimator 139/300, Train metric: 0.2393
Estimator 140/300, Train metric: 0.2393
Estimator 141/300, Train metric: 0.2393
Estimator 142/300, Train metric: 0.2393


       0.05       0.1      0.15       0.2       0.3       0.4       0.5  \
0 -0.660923 -0.411019 -0.057119  0.031250  0.135315  0.274188  0.341554   
1 -0.614034 -0.322651 -0.269790 -0.006447  0.133035  0.277592  0.387668   
2 -0.016880  0.253683  0.510851  0.586418  0.813666  0.908789  0.995536   
3  0.137018  0.320186  0.437684  0.559198  0.667784  0.750463  0.838678   
4 -0.496977 -0.293617  0.043573  0.093584  0.291570  0.381616  0.515648   

        0.6       0.7       0.8      0.85       0.9      0.95  Actual  
0  0.472242  0.595670  0.744184  0.888949  1.027598  1.375551     0.4  
1  0.496276  0.655190  0.915460  1.005439  1.166568  1.414633     0.2  
2  1.098232  1.202856  1.331882  1.458007  1.740235  2.063441     1.1  
3  0.904108  1.078036  1.345024  1.436642  1.474967  1.764087     1.6  
4  0.591708  0.727078  0.931764  1.022090  1.129131  1.327065     0.2  


In [33]:
# ACP execution for PGBM
best_params = get_best_model_params(best_scores_autosampler, "PGBM")
run_full_acp_pipeline(
    model_class=PGBMWrapper,
    best_params=best_params,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    model_name="PGBM",
    excel_file_path="./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/PGBM.xlsx",
)


Training on CPU
Estimator 0/300, Train metric: 1.2766
Estimator 1/300, Train metric: 1.2280
Estimator 2/300, Train metric: 1.1828
Estimator 3/300, Train metric: 1.1410
Estimator 4/300, Train metric: 1.0990
Estimator 5/300, Train metric: 1.0606
Estimator 6/300, Train metric: 1.0244
Estimator 7/300, Train metric: 0.9896
Estimator 8/300, Train metric: 0.9554
Estimator 9/300, Train metric: 0.9242
Estimator 10/300, Train metric: 0.8933
Estimator 11/300, Train metric: 0.8644
Estimator 12/300, Train metric: 0.8371
Estimator 13/300, Train metric: 0.8116
Estimator 14/300, Train metric: 0.7873
Estimator 15/300, Train metric: 0.7641


Estimator 16/300, Train metric: 0.7412
Estimator 17/300, Train metric: 0.7194


Estimator 18/300, Train metric: 0.6993
Estimator 19/300, Train metric: 0.6796
Estimator 20/300, Train metric: 0.6611
Estimator 21/300, Train metric: 0.6434
Estimator 22/300, Train metric: 0.6225
Estimator 23/300, Train metric: 0.6065
Estimator 24/300, Train metric: 0.5904
Estimator 25/300, Train metric: 0.5719
Estimator 26/300, Train metric: 0.5544
Estimator 27/300, Train metric: 0.5417
Estimator 28/300, Train metric: 0.5251
Estimator 29/300, Train metric: 0.5131
Estimator 30/300, Train metric: 0.5026
Estimator 31/300, Train metric: 0.4914
Estimator 32/300, Train metric: 0.4776
Estimator 33/300, Train metric: 0.4641
Estimator 34/300, Train metric: 0.4511
Estimator 35/300, Train metric: 0.4397
Estimator 36/300, Train metric: 0.4288
Estimator 37/300, Train metric: 0.4186
Estimator 38/300, Train metric: 0.4084
Estimator 39/300, Train metric: 0.4016
Estimator 40/300, Train metric: 0.3952
Estimator 41/300, Train metric: 0.3867


Estimator 42/300, Train metric: 0.3785
Estimator 43/300, Train metric: 0.3710


Estimator 44/300, Train metric: 0.3650
Estimator 45/300, Train metric: 0.3582
Estimator 46/300, Train metric: 0.3519
Estimator 47/300, Train metric: 0.3470
Estimator 48/300, Train metric: 0.3421
Estimator 49/300, Train metric: 0.3377
Estimator 50/300, Train metric: 0.3337
Estimator 51/300, Train metric: 0.3305
Estimator 52/300, Train metric: 0.3269
Estimator 53/300, Train metric: 0.3242
Estimator 54/300, Train metric: 0.3207
Estimator 55/300, Train metric: 0.3177
Estimator 56/300, Train metric: 0.3149
Estimator 57/300, Train metric: 0.3123
Estimator 58/300, Train metric: 0.3098
Estimator 59/300, Train metric: 0.3072
Estimator 60/300, Train metric: 0.3051
Estimator 61/300, Train metric: 0.3033
Estimator 62/300, Train metric: 0.3015
Estimator 63/300, Train metric: 0.2993
Estimator 64/300, Train metric: 0.2969
Estimator 65/300, Train metric: 0.2950
Estimator 66/300, Train metric: 0.2935
Estimator 67/300, Train metric: 0.2922
Estimator 68/300, Train metric: 0.2911
Estimator 69/300, Train m

Estimator 100/300, Train metric: 0.2557
Estimator 101/300, Train metric: 0.2557
Estimator 102/300, Train metric: 0.2557
Estimator 103/300, Train metric: 0.2557
Estimator 104/300, Train metric: 0.2557
Estimator 105/300, Train metric: 0.2557
Estimator 106/300, Train metric: 0.2557
Estimator 107/300, Train metric: 0.2557
Estimator 108/300, Train metric: 0.2557
Estimator 109/300, Train metric: 0.2557
Estimator 110/300, Train metric: 0.2556
Estimator 111/300, Train metric: 0.2556


Estimator 112/300, Train metric: 0.2556
Estimator 113/300, Train metric: 0.2556
Estimator 114/300, Train metric: 0.2556
Estimator 115/300, Train metric: 0.2556
Estimator 116/300, Train metric: 0.2556
Estimator 117/300, Train metric: 0.2556
Estimator 118/300, Train metric: 0.2556
Estimator 119/300, Train metric: 0.2556
Estimator 120/300, Train metric: 0.2556
Estimator 121/300, Train metric: 0.2556
Estimator 122/300, Train metric: 0.2556
Estimator 123/300, Train metric: 0.2556
Estimator 124/300, Train metric: 0.2556
Estimator 125/300, Train metric: 0.2556
Estimator 126/300, Train metric: 0.2556
Estimator 127/300, Train metric: 0.2556
Estimator 128/300, Train metric: 0.2556
Estimator 129/300, Train metric: 0.2556
Estimator 130/300, Train metric: 0.2556
Estimator 131/300, Train metric: 0.2556
Estimator 132/300, Train metric: 0.2556
Estimator 133/300, Train metric: 0.2556
Estimator 134/300, Train metric: 0.2556
Estimator 135/300, Train metric: 0.2556
Estimator 136/300, Train metric: 0.2556


PGBM best lambda: 50
PGBM best run: 1
PGBM ACP alpha_hat: 0.1226
PGBM ACP width: 5.4091
PGBM ACP coverage: 100.00%
PGBM ACP RMSE: 0.3710
PGBM ACP MAE: 0.2729
PGBM ACP target coverage (1 - alpha~): 87.74%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          4.938949       0.255051         0.744949            1.0 2.383764              0.0
   20.0                   2          7.094863       0.185175         0.814825            1.0 3.381206              0.0
   50.0                   2         11.563868       0.122569         0.877431            1.0 5.409115              0.0


{'model': PGBMWrapper(),
 'all_results': {10.0: {'all_losses': array([[10.511569 , 10.485132 , 10.445227 , ...,  4.939473 ,  4.9374294,
            4.942573 ],
          [10.509916 , 10.492235 , 10.469189 , ...,  4.936019 ,  4.9399843,
            4.9389486],
          [10.506962 , 10.486964 , 10.457617 , ...,  4.940624 ,  4.9406176,
            4.9400263]], dtype=float32),
   'all_sizes': array([[0.57934725, 0.5808962 , 0.582888  , ..., 2.3694334 , 2.350528  ,
           2.415846  ],
          [0.5792315 , 0.57966256, 0.58118284, ..., 2.3799326 , 2.3997416 ,
           2.3604617 ],
          [0.57944095, 0.581016  , 0.58242214, ..., 2.3383172 , 2.4018226 ,
           2.3862884 ]], dtype=float32),
   'all_alphas': array([[0.9932221 , 0.99042356, 0.98623383, ..., 0.25700393, 0.25869015,
           0.2526727 ],
          [0.9930685 , 0.9912573 , 0.98880064, ..., 0.25560862, 0.2540243 ,
           0.25784868],
          [0.9927521 , 0.99059474, 0.9875195 , ..., 0.26023072, 0.2538795 ,
   

# **Conformal Predictions with TabNet**


In [34]:
best_params = get_best_model_params(best_scores_autosampler, 'TabNet')
print(best_params)


{'n_d': 8, 'n_a': 64, 'n_steps': 3, 'gamma': 1.5, 'lambda_sparse': 0.001, 'optimizer_params': {'lr': 0.02}, 'mask_type': 'entmax', 'n_shared': 2, 'n_independent': 1, 'scheduler_params': {'step_size': 10, 'gamma': 0.9}, 'scheduler_fn': <class 'torch.optim.lr_scheduler.StepLR'>, 'seed': 42, 'verbose': 0}


In [35]:
class TabNetRegressorCP(TabNetRegressor):
    def fit(self, X, y, *args, **kwargs):
        y_arr = np.asarray(y)
        if y_arr.ndim == 1:
            y_arr = y_arr.reshape(-1, 1)
        return super().fit(X, y_arr, *args, **kwargs)

    def predict(self, X, *args, **kwargs):
        preds = super().predict(X, *args, **kwargs)
        return np.asarray(preds).reshape(-1)


best_params = get_best_model_params(best_scores_autosampler, "TabNet")
if isinstance(best_params, dict) and "verbose" in best_params:
    best_params = dict(best_params)
    best_params.pop("verbose", None)

run_full_acp_pipeline(
    TabNetRegressorCP,
    best_params,
    X_train,
    y_train,
    X_test,
    y_test,
    "TabNet",
    "./drive/MyDrive/Regression/scour_uncertainty_analysis/conformal_predictions_adaptive_coverage_policies/TabNet.xlsx",
)


epoch 0  | loss: 2.50162 |  0:00:00s
epoch 1  | loss: 1.96379 |  0:00:00s
epoch 2  | loss: 0.82485 |  0:00:00s
epoch 3  | loss: 0.50282 |  0:00:00s


epoch 4  | loss: 0.33105 |  0:00:00s
epoch 5  | loss: 0.53572 |  0:00:00s
epoch 6  | loss: 0.37867 |  0:00:00s
epoch 7  | loss: 0.31543 |  0:00:00s
epoch 8  | loss: 0.23488 |  0:00:00s
epoch 9  | loss: 0.17654 |  0:00:00s
epoch 10 | loss: 0.13557 |  0:00:00s


epoch 11 | loss: 0.12549 |  0:00:00s
epoch 12 | loss: 0.11655 |  0:00:00s
epoch 13 | loss: 0.11229 |  0:00:00s
epoch 14 | loss: 0.10436 |  0:00:00s


epoch 15 | loss: 0.08232 |  0:00:00s
epoch 16 | loss: 0.0818  |  0:00:00s
epoch 17 | loss: 0.10092 |  0:00:00s
epoch 18 | loss: 0.10156 |  0:00:00s
epoch 19 | loss: 0.07225 |  0:00:00s
epoch 20 | loss: 0.06111 |  0:00:00s
epoch 21 | loss: 0.0635  |  0:00:00s


epoch 22 | loss: 0.07398 |  0:00:00s
epoch 23 | loss: 0.09328 |  0:00:00s
epoch 24 | loss: 0.08492 |  0:00:00s
epoch 25 | loss: 0.07229 |  0:00:00s


epoch 26 | loss: 0.06898 |  0:00:00s
epoch 27 | loss: 0.06461 |  0:00:00s
epoch 28 | loss: 0.06597 |  0:00:00s
epoch 29 | loss: 0.06666 |  0:00:00s
epoch 30 | loss: 0.06168 |  0:00:00s
epoch 31 | loss: 0.05566 |  0:00:00s


... (12 outputs left out)

epoch 95 | loss: 0.0094  |  0:00:02s
epoch 96 | loss: 0.00927 |  0:00:02s
epoch 97 | loss: 0.00917 |  0:00:02s
epoch 98 | loss: 0.00905 |  0:00:02s
epoch 99 | loss: 0.00891 |  0:00:02s


TabNet best lambda: 50
TabNet best run: 1
TabNet ACP alpha_hat: 0.1541
TabNet ACP width: 6.9601
TabNet ACP coverage: 100.00%
TabNet ACP RMSE: 0.5434
TabNet ACP MAE: 0.3741
TabNet ACP target coverage (1 - alpha~): 84.59%  |  miss / alpha~: 0.000  # FIX A1


 lambda  best_run (1-based)  final_train_loss  alpha_learned  Target_coverage  test_coverage    width  Miss_over_alpha
   10.0                   2          6.334992       0.324771         0.675229       0.987179 3.085382         0.039476
   20.0                   2          9.069555       0.234707         0.765293       1.000000 4.368757         0.000000
   50.0                   2         14.687288       0.154092         0.845908       1.000000 6.960092         0.000000


{'model': TabNetRegressorCP(n_d=8, n_a=64, n_steps=3, gamma=1.5, cat_idxs=[], cat_dims=[], cat_emb_dim=[], n_independent=1, n_shared=2, epsilon=1e-15, momentum=0.02, lambda_sparse=0.001, seed=42, clip_value=1, verbose=1, optimizer_fn=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.02}, scheduler_fn=<class 'torch.optim.lr_scheduler.StepLR'>, scheduler_params={'step_size': 10, 'gamma': 0.9}, mask_type='entmax', input_dim=7, output_dim=1, device_name='auto', n_shared_decoder=1, n_indep_decoder=1, grouped_features=[], compile_backend=''),
 'all_results': {10.0: {'all_losses': array([[10.901857 , 10.877959 , 10.838541 , ...,  6.334571 ,  6.3370304,
            6.337096 ],
          [10.900449 , 10.883862 , 10.863029 , ...,  6.337319 ,  6.3396764,
            6.3349924],
          [10.8984165, 10.879501 , 10.8508835, ...,  6.3389664,  6.3351564,
            6.338615 ]], dtype=float32),
   'all_sizes': array([[0.9696196 , 0.97375137, 0.97616374, ..., 3.078142  , 3.0513105 ,
       